# Mercury - Max Tier Fine-Tuning (Thoth 1.3)

Covers Thoth's two engines: Parakeet-TDT-0.6B-v2 (ASR, NeMo) and CosyVoice2-0.5B (TTS).
Full fine-tune and LoRA/PEFT/adapter paths for each - see MODELVERSION.md for what a fine-tune does to the version number (1.3 -> 2.0 on first fine-tune, then 2.0 -> 2.1 -> 2.2 for each one after).

These are the heaviest two models in the catalog - expect the install cell and the two
from_pretrained calls to take noticeably longer than the Free/Pro notebooks.

## 0. Setup - install dependencies

In [ ]:
!pip install -q nemo_toolkit[asr]==2.0.0 pytorch-lightning==2.2.5 "datasets<3" torch==2.5.1 torchaudio==2.5.1 peft accelerate soundfile pydub jiwer huggingface_hub

# CosyVoice2 isn't a pip package - clone its repo for the fine-tune recipe it ships.
!git clone -q --recursive https://github.com/FunAudioLLM/CosyVoice.git /kaggle/working/CosyVoice || true

## 1. Convert any WAV source audio to MP3

Run once against your raw dataset directory before building manifests - keeps disk
quota and any exported archives small. Skip if your data is already MP3.

In [ ]:
import os
from pydub import AudioSegment

DATASET_DIR = "/kaggle/input/your-dataset"  # <-- point at your real dataset
OUT_DIR = "/kaggle/working/audio_mp3"
os.makedirs(OUT_DIR, exist_ok=True)

converted = 0
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.lower().endswith(".wav"):
            src = os.path.join(root, f)
            dst = os.path.join(OUT_DIR, os.path.splitext(f)[0] + ".mp3")
            AudioSegment.from_wav(src).export(dst, format="mp3", bitrate="128k")
            converted += 1
print(f"Converted {converted} wav files to mp3 in {OUT_DIR}")

## 2. Build train/val manifests

NeMo-format manifests (audio_filepath/text/duration per line) - the NeMo ASR model
below consumes this format directly.

In [ ]:
import json
import soundfile as sf

MANIFEST_TRAIN = "/kaggle/working/train_manifest.json"
MANIFEST_VAL = "/kaggle/working/val_manifest.json"

def build_manifest(audio_text_pairs, out_path):
    with open(out_path, "w") as f:
        for audio_path, text in audio_text_pairs:
            info = sf.info(audio_path)
            f.write(json.dumps({
                "audio_filepath": audio_path,
                "text": text,
                "duration": info.duration,
            }) + "\n")

# TODO: populate from your real (audio_path, transcript) pairs once data is ready.
train_pairs = []
val_pairs = []
build_manifest(train_pairs, MANIFEST_TRAIN)
build_manifest(val_pairs, MANIFEST_VAL)
print(f"train={len(train_pairs)} val={len(val_pairs)} examples (0 until real data is wired in)")

---
## 3. ASR - Parakeet-TDT-0.6B-v2 (NeMo adapter/PEFT path)

Uses NeMo's native PEFT/adapter support rather than a hand-rolled LoRA wrapper.

In [ ]:
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")

# NeMo's adapter/PEFT config - attach once manifests are real; see
# nemo.collections.common.parts.adapter_modules for the adapter types available.
# asr_model.add_adapter(name="lora_adapter", cfg=<AdapterConfig>)

print(asr_model.cfg.train_ds if hasattr(asr_model.cfg, "train_ds") else "no train_ds in cfg yet")

### 3b. ASR - full fine-tune path (no adapter)

Unfreezes every parameter via NeMo's standard Trainer/setup_training_data flow instead
of an adapter - substantially more VRAM/time than the adapter path above.

In [ ]:
import nemo.collections.asr as nemo_asr

asr_model_full = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
asr_model_full.train()
print(f"full fine-tune: {sum(p.numel() for p in asr_model_full.parameters() if p.requires_grad):,} trainable params")

---
## 4. TTS - CosyVoice2-0.5B

Official repo (cloned in step 0) ships its own fine-tune recipe under CosyVoice/examples/ -
check whether it exposes a LoRA/adapter mode for your CosyVoice version; if only full-param
fine-tuning is offered upstream, freeze most layers manually as a low-rank-equivalent.

In [ ]:
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # see spaces/space-max/app.py for why

import sys
sys.path.insert(0, "/kaggle/working/CosyVoice")
sys.path.insert(0, "/kaggle/working/CosyVoice/third_party/Matcha-TTS")

from huggingface_hub import snapshot_download
from cosyvoice.cli.cosyvoice import CosyVoice2

# CosyVoice2(model_dir) calls ModelScope's own snapshot_download if handed a bare repo
# id, which 404s (different registry than HF, doesn't have this repo) - pre-fetch from
# HF ourselves and hand it a local dir instead. See spaces/space-max/app.py for the
# same fix applied to the serving path.
local_dir = snapshot_download("FunAudioLLM/CosyVoice2-0.5B")
cosyvoice = CosyVoice2(local_dir)
# See CosyVoice/examples/ for the official fine-tune recipe once real data is ready.

---
## 5. Evaluation harness (run after any fine-tune, before promoting a model)

In [ ]:
from jiwer import wer, cer

def evaluate(model_transcribe_fn, val_pairs):
    refs, hyps = [], []
    for audio_path, ref_text in val_pairs:
        hyps.append(model_transcribe_fn(audio_path))
        refs.append(ref_text)
    if not refs:
        return {"wer": None, "cer": None, "n": 0}
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "n": len(refs)}

# evaluate(lambda path: ..., val_pairs)  # wire up the real transcribe fn once training runs